In [2]:
import hashlib
from cryptography.hazmat.primitives.asymmetric import ec
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import utils
from cryptography.exceptions import InvalidSignature
import binascii

# --- 1. Define the Transaction Message ---
transaction_message = "Ahmed pays 100 coins to Ali"
print(f"Transaction Message: {transaction_message}")

# --- 2. Generate an ECDSA Private-Public Key Pair (P-256 Curve) ---
# A private key is generated using the secp256r1 curve (a standard ECDSA curve)
private_key = ec.generate_private_key(
    ec.SECP256R1()
)
public_key = private_key.public_key()
print("\n--- Key Generation Successful ---")
# Optional: Display keys (truncated for security/brevity)
# print(f"Private Key (Snippet): {private_key.private_numbers().private_value:x}"[:20] + "...")
# print(f"Public Key (Snippet): {public_key.public_numbers().x:x}"[:20] + "...")


# --- 3. Hash the message using SHA-256 ---
# The message is converted to bytes before hashing
message_bytes = transaction_message.encode('utf-8')
hasher = hashes.Hash(hashes.SHA256())
hasher.update(message_bytes)
hash_digest = hasher.finalize()

print(f"\n--- Hashing Successful ---")
# Convert the hash digest to a hexadecimal string for display
hash_hex = binascii.hexlify(hash_digest).decode('utf-8')
print(f"SHA-256 Hash: {hash_hex}")


# --- 4. Sign the hash using the private key (ECDSA) ---
# The signature process uses the SHA-256 algorithm and the DER encoding for the signature.
signature = private_key.sign(
    hash_digest,
    ec.ECDSA(hashes.SHA256())
)

print(f"\n--- Signing Successful ---")
# Convert the signature to a hexadecimal string for display
signature_hex = binascii.hexlify(signature).decode('utf-8')
# Print a snippet of the signature
print(f"Signature: {signature_hex[:25]}...")


# --- 5. Verify the signature using the public key ---
verification_result = False
try:
    # Use the public key to verify the signature against the original hash digest
    public_key.verify(
        signature,
        hash_digest,
        ec.ECDSA(hashes.SHA256())
    )
    verification_result = True
except InvalidSignature:
    verification_result = False

# --- 6. Print results and analyze ---
print(f"\n--- Verification Result ---")
print(f"Signature Verified: {verification_result}")

# --- Demonstration of what happens if the message/hash changes (for Q5) ---
print("\n--- Tampering Test ---")
tampered_message_bytes = "Ahmed pays 100 coins to Ali, but changed his mind.".encode('utf-8')
tampered_hasher = hashes.Hash(hashes.SHA256())
tampered_hasher.update(tampered_message_bytes)
tampered_hash_digest = tampered_hasher.finalize()

tampering_result = False
try:
    public_key.verify(
        signature, # The original signature
        tampered_hash_digest, # The new, tampered hash
        ec.ECDSA(hashes.SHA256())
    )
    tampering_result = True
except InvalidSignature:
    tampering_result = False

print(f"Verification with Tampered Message: {tampering_result}") # This should be False

Transaction Message: Ahmed pays 100 coins to Ali

--- Key Generation Successful ---

--- Hashing Successful ---
SHA-256 Hash: 3f290af40555abd9fdf58239d8eac3b8e5df5c07f36951285df7e4356496d7be

--- Signing Successful ---
Signature: 3046022100d470eec8b5fe6fc...

--- Verification Result ---
Signature Verified: True

--- Tampering Test ---
Verification with Tampered Message: False
